In [ ]:
import itertools

import clifft
import stim
from IPython.core.display import Markdown

from library.circuitry import Circuitry
from library.common import Pauli
from library.qubit_array import QubitArray
from utils.simulation.clifft import sample

In [ ]:
def format(preparation: Pauli, observable: Pauli):
    return rf"$\overline{{\mathbf{{{preparation.name}}}}}/\overline{{\mathbf{{{observable.name}}}}}$"

In [ ]:
display_url = False

In [ ]:
scenarios_direct = {}

for preparation, observable in itertools.product(Pauli, repeat=2):
    qubits = QubitArray(dimensions=(1,1))
    circuitry = Circuitry(qubits, clifford=False)

    circuitry.append(f"R{preparation.name}", 0)
    circuitry.append("T", 0)

    circuitry.append_observable(0, "OBSERVABLE", { 0: observable.name })

    scenario = rf"$\overline{{\mathbf{{{preparation.name}}}}}/\overline{{\mathbf{{{observable.name}}}}}$"
    scenarios_direct[scenario] = circuitry

    if display_url:
        display(Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url(location="http://localhost:8000/crumble.html")})"))

In [ ]:
scenarios_feedforward = {}

for preparation, observable in itertools.product(Pauli, repeat=2):
    qubits = QubitArray(dimensions=(1, 3))
    circuitry = Circuitry(qubits, clifford=False)

    circuitry.append(f"R{preparation.name}", 0)
    circuitry.append("RX", [1, 2])
    circuitry.append("S", 1)
    circuitry.append("T", 2)
    circuitry.append("CX", [2, 0])

    circuitry.append("MZ", 0)
    qubits.record_measurement(0, "M0")

    circuitry.append("CX", [stim.target_rec(-1), 2])

    # Perform the S-correction
    circuitry.append("CX", [2, 1])

    # Perform the Conditional-H, feed-forward style
    # cfr. https://unitaryfoundation.github.io/clifft/stable/reference/gates/#rewrite-gates
    circuitry.append("R_Y", 1, arg=+0.25)
    circuitry.append("CX", [stim.target_rec(-1), 1])
    circuitry.append("R_Y", 1, arg=-0.25)

    circuitry.append("MX", 1)
    qubits.record_measurement(1, "M1")

    circuitry.append("CZ", [stim.target_rec(-1), 2])
    circuitry.append_tick()

    circuitry.append_observable(0, "OBSERVABLE", { 2: observable.name })

    scenario = rf"$\overline{{\mathbf{{{preparation.name}}}}}/\overline{{\mathbf{{{observable.name}}}}}$"
    scenarios_feedforward[scenario] = circuitry

    # print(str(circuitry.as_clifft_text))

    if display_url:
        display(Markdown(f"[Open in Crumble ({scenario})]({circuitry.to_crumble_url(location="http://localhost:8000/crumble.html")})"))

In [ ]:
title = rf"$\mathbf{{T}}$-gate direct"
sample(scenarios_direct, correction=True, title = title, label="Preparation/Observable", shots=1e6, fontsize=8, figsize=(16, 5))

In [ ]:
title = rf"$\mathbf{{T}}$-gate construct [feed-forward]"
sample(scenarios_feedforward, postselection=True, correction=True, title = title, label="Preparation/Observable", shots=1e6, fontsize=8, figsize=(16, 5))